# 레슨 04 — 실습 문제 정답지

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/04/%EB%A0%88%EC%8A%A8%2004%20%E2%80%94%20%ED%8C%8C%EC%9D%BC%20%EB%8B%A4%EC%9A%B4%EB%A1%9C%EB%93%9C%EC%99%80%20%ED%8F%B4%EB%8D%94%20%EC%A0%95%EB%A6%AC.ipynb)

파일 다운로드와 폴더 정리 실습 문제의 모범 답안이다. 출력만 맞는지보다 경로 처리, 파일명 정리, 저장 검증, 로그 산출물이 함께 갖춰졌는지 확인한다.

## 0. 환경 셀


In [ ]:
import os
import re
import csv
import json
import shutil
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/04/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def read_soup(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def safe_filename(name):
    base = Path(str(name)).name
    cleaned = re.sub(r'[^0-9A-Za-z가-힣_.-]+', '_', base).strip('._')
    return cleaned or 'downloaded_file'

def save_bytes(relative_path, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    filename = safe_filename(relative_path)
    target = target_dir / filename
    target.write_bytes(load_bytes(relative_path))
    return target

def file_info(path):
    path = Path(path)
    return {'path': str(path), 'name': path.name, 'size': path.stat().st_size, 'exists': path.exists()}

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 정답 — 자료실 HTML 열기


In [ ]:
html_text = load_text('resource_center.html')
soup = BeautifulSoup(html_text, 'html.parser')
print(soup.select_one('h1').text.strip())


### 왜 이 코드가 정답인지

load_text로 fixture를 읽어야 코랩과 로컬 경로 차이를 줄일 수 있다. BeautifulSoup으로 파싱한 뒤 h1을 읽으면 자료실 HTML이 정상적으로 열렸는지 확인할 수 있다.

### 채점 포인트

- HTML 링크 개수와 manifest 행 수가 맞는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- href 대신 화면 텍스트만 저장 대상으로 쓰는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 2 정답 — 파일 링크 개수 세기


In [ ]:
links = soup.select('a.file-link')
print('links:', len(links))


### 왜 이 코드가 정답인지

a.file-link는 자료실의 파일 링크만 선택하는 selector다. 개수를 먼저 확인하면 링크 누락이나 selector 오타를 빠르게 잡을 수 있다.

### 채점 포인트

- 상대 경로를 원격 URL 또는 로컬 Path로 안전하게 바꾸었는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- DATA_BASE와 href를 문자열 덧셈으로 붙여 슬래시가 깨지는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 3 정답 — 첫 링크 정보 읽기


In [ ]:
first = links[0]
print(first.text.strip())
print(first['href'])
print(first['data-type'], first['data-group'])


### 왜 이 코드가 정답인지

다운로드 대상은 화면 텍스트만으로 충분하지 않다. href는 실제 읽을 경로이고 data-type, data-group은 저장 분류 기준이므로 함께 읽어야 한다.

### 채점 포인트

- 저장 파일명이 운영체제에서 안전하게 처리되는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- manifest의 path가 아니라 file_name으로 파일을 읽어 경로 오류가 나는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 4 정답 — 소스 위치 만들기


In [ ]:
href = first['href']
if DATA_BASE.startswith('http'):
    print(urljoin(DATA_BASE + '/', href))
else:
    print(Path(DATA_BASE) / href)


### 왜 이 코드가 정답인지

상대 경로는 실행 환경에 따라 해석 방식이 달라진다. urljoin과 Path를 나누어 사용하면 코랩과 로컬에서 모두 올바른 위치를 만들 수 있다.

### 채점 포인트

- 저장된 파일 크기가 0보다 큰지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 폴더를 만들지 않고 write_bytes를 호출해 실패하는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 5 정답 — manifest 읽기


In [ ]:
manifest = list(csv.DictReader(load_text('manifest.csv').splitlines()))
for row in manifest[:3]:
    print(row['file_name'], row['group'])


### 왜 이 코드가 정답인지

manifest는 다운로드 목록의 기준표다. DictReader로 읽으면 file_name, group, type, path를 key로 접근할 수 있어 HTML 링크와 비교하기 쉽다.

### 채점 포인트

- CSV 또는 JSON 산출물을 다시 읽어 검증했는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 로그 파일에 header를 쓰지 않아 검수하기 어려운 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 6 정답 — 확장자별 개수 세기


In [ ]:
type_counts = {}
for row in manifest:
    key = row['type']
    type_counts[key] = type_counts.get(key, 0) + 1
print(type_counts)


### 왜 이 코드가 정답인지

다운로드 전에 유형별 개수를 알면 저장 후 결과 검증이 쉬워진다. type 컬럼은 확장자 기준 요약을 만드는 가장 직접적인 값이다.

### 채점 포인트

- HTML 링크 개수와 manifest 행 수가 맞는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- href 대신 화면 텍스트만 저장 대상으로 쓰는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 7 정답 — 안전한 파일명 만들기


In [ ]:
names = ['week 1 자료.txt', 'score/template.csv', '학생용 체크리스트.md']
for name in names:
    print(name, '->', safe_filename(name))


### 왜 이 코드가 정답인지

저장 파일명은 운영체제와 로그에서 안정적으로 다뤄져야 한다. safe_filename을 사용하면 경로 구분자와 불필요한 특수문자를 제거할 수 있다.

### 채점 포인트

- 상대 경로를 원격 URL 또는 로컬 Path로 안전하게 바꾸었는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- DATA_BASE와 href를 문자열 덧셈으로 붙여 슬래시가 깨지는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 8 정답 — 하나의 파일 저장하기


In [ ]:
target = save_bytes('files/lesson_plan.md', 'downloads/lesson04/single')
print(target)
print(target.stat().st_size)


### 왜 이 코드가 정답인지

파일 저장은 경로가 만들어지는 것과 실제 bytes가 쓰이는 것을 모두 확인해야 한다. stat().st_size가 0보다 크면 기본 저장은 성공한 것이다.

### 채점 포인트

- 저장 파일명이 운영체제에서 안전하게 처리되는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- manifest의 path가 아니라 file_name으로 파일을 읽어 경로 오류가 나는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 9 정답 — HTML 링크 전체 저장하기


In [ ]:
saved = []
for link in links:
    saved.append(save_bytes(link['href'], 'downloads/lesson04/all'))
print(len(saved))
print(saved[0])


### 왜 이 코드가 정답인지

HTML 링크 전체 저장은 반복 다운로드의 기본 구조다. href를 기준으로 저장하고 결과 리스트 길이를 확인해야 누락 여부를 알 수 있다.

### 채점 포인트

- 저장된 파일 크기가 0보다 큰지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 폴더를 만들지 않고 write_bytes를 호출해 실패하는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 10 정답 — 그룹별 폴더로 저장하기


In [ ]:
grouped = []
for row in manifest:
    target_dir = Path('downloads/lesson04/by_group') / row['group']
    grouped.append(save_bytes(row['path'], target_dir))
print(grouped[:3])


### 왜 이 코드가 정답인지

group 기준 저장은 운영자가 자료를 찾기 쉽게 만든다. manifest의 path는 원본 파일 경로이고 group은 저장 폴더 기준으로 사용한다.

### 채점 포인트

- CSV 또는 JSON 산출물을 다시 읽어 검증했는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 로그 파일에 header를 쓰지 않아 검수하기 어려운 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 11 정답 — 저장 파일 크기 검증하기


In [ ]:
empty_files = []
for target in grouped:
    if target.stat().st_size == 0:
        empty_files.append(target)
print('empty:', len(empty_files), 'total:', len(grouped))


### 왜 이 코드가 정답인지

다운로드 자동화는 파일이 존재하는지만 보면 부족하다. 크기가 0인 파일은 실패로 봐야 하므로 저장 직후 size 검증이 필요하다.

### 채점 포인트

- HTML 링크 개수와 manifest 행 수가 맞는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- href 대신 화면 텍스트만 저장 대상으로 쓰는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 12 정답 — 다운로드 로그 만들기


In [ ]:
download_log = []
for row, target in zip(manifest, grouped):
    download_log.append({'file_name': row['file_name'], 'group': row['group'], 'saved_path': str(target), 'size': target.stat().st_size})
print(download_log[0])
print(len(download_log))


### 왜 이 코드가 정답인지

로그는 원본 정보와 저장 결과를 이어 준다. file_name, group, saved_path, size가 있으면 나중에 누락과 빈 파일을 점검할 수 있다.

### 채점 포인트

- 상대 경로를 원격 URL 또는 로컬 Path로 안전하게 바꾸었는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- DATA_BASE와 href를 문자열 덧셈으로 붙여 슬래시가 깨지는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 13 정답 — 로그 CSV 저장하기


In [ ]:
log_path = Path('downloads/lesson04/download_log.csv')
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['file_name', 'group', 'saved_path', 'size'])
    writer.writeheader()
    writer.writerows(download_log)
print(log_path, len(download_log))


### 왜 이 코드가 정답인지

CSV 로그는 자동화 결과를 재검토하는 기준이다. 헤더를 먼저 쓰고 모든 로그 행을 저장해야 사람이 열어도 의미가 분명하다.

### 채점 포인트

- 저장 파일명이 운영체제에서 안전하게 처리되는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- manifest의 path가 아니라 file_name으로 파일을 읽어 경로 오류가 나는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 14 정답 — 그룹별 저장 개수 검증하기


In [ ]:
loaded_log = list(csv.DictReader(log_path.read_text(encoding='utf-8').splitlines()))
group_counts = Counter(row['group'] for row in loaded_log)
print(dict(group_counts))


### 왜 이 코드가 정답인지

저장 로그를 다시 읽는 검증은 산출물이 실제로 남았는지 확인하는 과정이다. group별 개수가 manifest와 맞으면 폴더 분류가 정상이다.

### 채점 포인트

- 저장된 파일 크기가 0보다 큰지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 폴더를 만들지 않고 write_bytes를 호출해 실패하는 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 문제 15 정답 — 다운로드 요약 JSON 저장하기


In [ ]:
summary = {'files': len(download_log), 'types': dict(type_counts), 'empty': len(empty_files)}
summary_path = Path('downloads/lesson04/download_summary.json')
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(summary)
print(summary_path)


### 왜 이 코드가 정답인지

CSV는 행 단위 로그에 적합하고 JSON은 전체 요약을 담기 좋다. 파일 수, 유형별 개수, 빈 파일 수를 저장하면 다운로드 결과를 빠르게 확인할 수 있다.

### 채점 포인트

- CSV 또는 JSON 산출물을 다시 읽어 검증했는지 확인한다.
- 경로, 파일 수, 로그 행 수가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 로그 파일에 header를 쓰지 않아 검수하기 어려운 경우가 있다.
- 결과는 비슷하지만 파일이 실제로 저장되지 않았다.
- 코랩과 로컬 중 한쪽에서만 동작하는 경로를 사용했다.

---

## 교사용 상세 피드백 기준

### 링크 추출 단계

문제 1~4에서는 HTML에서 어떤 값을 읽는지 정확히 구분해야 한다. 파일명은 텍스트, 실제 경로는 href, 분류 기준은 data-type과 data-group에 들어 있다. 학생이 이 구분을 못 하면 이후 저장 폴더를 만들 때 화면 문자열과 경로 문자열을 섞게 된다.

### 저장 단계

문제 7~10에서는 파일명이 안전한지, 폴더가 만들어지는지, bytes가 실제로 쓰이는지 확인한다. 파일 저장 함수가 경로를 반환하면 다음 단계에서 로그를 만들기 쉽다. 저장 경로를 문자열로만 출력하고 Path 객체를 반환하지 않으면 검증 코드가 길어진다.

### 검증 단계

문제 11~15는 다운로드 자동화의 핵심이다. 파일 존재 여부만 보면 부족하고, size가 0보다 큰지 봐야 한다. CSV 로그와 JSON 요약을 다시 읽는 과정까지 있어야 산출물이 실제로 남았는지 확인할 수 있다.

## 문제별 빠른 확인표

| 문제 | 핵심 검수 | 통과 기준 |
|---:|---|---|
| 1 | HTML 로드 | 제목 출력 |
| 2 | 링크 selector | 링크 개수 출력 |
| 3 | href와 data 속성 | 파일명, 경로, 분류 출력 |
| 4 | 환경별 경로 | URL 또는 Path 출력 |
| 5 | manifest CSV | 파일명과 그룹 출력 |
| 6 | 유형별 집계 | type_counts 출력 |
| 7 | 파일명 정리 | 안전한 이름 출력 |
| 8 | 단일 저장 | 파일 크기 출력 |
| 9 | 전체 링크 저장 | 저장 개수 확인 |
| 10 | 그룹 폴더 저장 | 그룹별 경로 생성 |
| 11 | 크기 검증 | 빈 파일 0개 확인 |
| 12 | 로그 rows | 원본과 저장 결과 연결 |
| 13 | CSV 로그 | header와 행 저장 |
| 14 | 로그 재검증 | group별 개수 확인 |
| 15 | JSON 요약 | files, types, empty 저장 |

## 재실행 검수 기준

교사용 검수에서는 downloads/lesson04 폴더를 삭제한 뒤 새 런타임에서 다시 실행해 본다. 같은 파일명, 같은 행 수, 같은 empty 값이 나오면 안정적이다. 저장 결과가 이전 실행에 의존하면 실제 운영 자동화로 보기 어렵다.

## 문제별 오답 대응 가이드

### 문제 1~3

HTML을 읽지 못하면 대부분 파일명 오타 또는 DATA_BASE 문제다. 학생이 soup 변수를 만들기 전에 load_text 결과를 출력해 보게 한다. links가 0이면 a 태그는 있지만 class를 틀렸는지 확인한다. href와 data 속성은 대괄호 접근을 사용해야 하며, text.strip()과 혼동하지 않게 지도한다.

### 문제 4~6

원격과 로컬 경로를 같은 방식으로 처리하려는 답안은 코랩에서 깨질 수 있다. DATA_BASE.startswith('http') 조건으로 분기하는 이유를 설명하게 한다. manifest는 CSV이므로 DictReader를 사용하고, row['path']가 실제 저장 대상이라는 점을 반복 확인한다. type_counts 결과가 비어 있으면 manifest를 먼저 제대로 읽었는지 본다.

### 문제 7~10

safe_filename은 원본 파일명 전체를 바꾸는 것이 아니라 저장용 이름을 만드는 함수다. 원본 이름은 로그에 남기고 저장 경로에는 안전한 이름을 쓰는 방식이 좋다. save_bytes는 target_dir을 만들고 파일을 쓴 뒤 Path를 반환해야 한다. 학생이 함수 안에서 print만 하면 다음 검증 단계에서 재사용하기 어렵다.

### 문제 11~15

파일 존재와 파일 크기 검증을 구분한다. 파일이 있어도 0 byte이면 실패로 본다. download_log는 리스트 상태에서 먼저 길이와 첫 행을 확인한 뒤 CSV로 저장하게 한다. CSV를 저장한 뒤 다시 읽는 문제는 산출물 검증을 위한 단계다. JSON 요약에는 Path 객체를 직접 넣지 말고 문자열, 숫자, 딕셔너리처럼 직렬화 가능한 값만 넣어야 한다.

## 엄격 채점이 필요한 항목

- 실제 외부 URL을 요청한 답안은 통과시키지 않는다.
- downloads/lesson04 밖에 저장하는 답안은 보완을 요구한다.
- 파일 저장 없이 경로 문자열만 만든 답안은 저장 문제 통과로 보지 않는다.
- size 검증이 없는 최종 미션은 운영 자동화로 보지 않는다.
- CSV 로그와 JSON 요약 중 하나라도 빠지면 최종 미션 완료로 보지 않는다.

## 우수 답안 기준

우수 답안은 저장 함수를 작게 분리하고, 실패를 status와 error로 남긴다. group 기준 외에 week 또는 type 기준 저장도 추가할 수 있다. 중복 방지를 위해 이미 저장된 파일의 크기를 비교하거나, manifest path를 set으로 관리하는 방식도 좋다. 단, 추가 기능이 들어가도 기본 산출물 이름과 구조는 과제 요구사항을 따라야 한다.

## 산출물 이름과 경로 검수

최종적으로 학생 폴더에는 downloads/lesson04 아래에 결과가 남아야 한다. 단일 저장 문제는 single 폴더, 전체 링크 저장 문제는 all 폴더, 그룹 저장 문제는 by_group 폴더를 사용한다. 최종 미션에서는 final 폴더를 사용해 연습 문제 산출물과 섞이지 않게 하는 것도 좋은 방식이다.

검수할 때는 경로 문자열에 downloads/lesson04가 포함되어 있는지 먼저 본다. 임시 시스템 폴더나 홈 디렉터리에 직접 저장하는 답안은 운영 수업 환경에서 찾기 어렵고, 다른 자료를 덮어쓸 위험이 있다. 저장 위치를 제한하는 것은 편의가 아니라 안전 기준이다.

## 로그 컬럼 권장안

기본 문제에서는 file_name, group, saved_path, size만 사용해도 된다. 최종 미션에서는 type, week, source_path, status까지 포함하는 편이 좋다. source_path는 다시 다운로드할 때 필요하고, status는 실패와 빈 파일을 구분하는 데 필요하다. error 컬럼을 추가하면 예외 발생 시 원인을 기록할 수 있다.

로그 컬럼이 많아져도 핵심은 행 수다. manifest 행 수와 로그 행 수가 같고, status가 모두 saved이며, size가 모두 0보다 크면 기본 성공이다. 이 세 조건을 학생이 말로 설명할 수 있으면 다운로드 자동화의 검증 흐름을 이해한 것으로 본다.
